# EXP-M — same protocol, different architecture: Phi-3.5-vision

Re-run the text and image TTFT sweeps against **microsoft/Phi-3.5-vision-instruct** (4.2B; CLIP ViT-L encoder with HD crops — a different vision architecture family than Qwen's native-resolution ViT).

Questions: is the ~3.8× image-vs-text per-token cost Qwen-specific or general? Does encoder overhead scale with resolution the same way when images are processed as fixed-size crops?

Note: Phi has **no closed-form token formula** — token counts are read from the server's own metering (that's why the harness always uses `usage.prompt_tokens`).

Runtime → **T4 GPU**. ~45 min (8.4 GB of weights).


In [ ]:
!nvidia-smi -L


In [ ]:
# Install vLLM 0.25.1 built for CUDA 12.9 (PyPI default targets CUDA 13 and fails on Colab)
%pip -q install https://github.com/vllm-project/vllm/releases/download/v0.25.1/vllm-0.25.1+cu129-cp38-abi3-manylinux_2_28_x86_64.whl --extra-index-url https://download.pytorch.org/whl/cu129
import vllm, transformers
print('vllm', vllm.__version__, '| transformers', transformers.__version__)


## Benchmark code (mirrors `bench/` in the repo)


In [ ]:
import base64, io, json, subprocess, time
import numpy as np
import requests
from PIL import Image
from transformers import AutoTokenizer

PHI = 'microsoft/Phi-3.5-vision-instruct'
RESULTS = '/content/results_exp_m.jsonl'

import base64
import io

import numpy as np
from PIL import Image


def image_tokens(width: int, height: int) -> int:
    """Tokens an image costs in Qwen2-VL: one per 28x28 pixel block."""
    return (height // 28) * (width // 28)


def noise_image(side_px: int, seed: int) -> Image.Image:
    """A square RGB random-noise image. Noise (not a flat color) so the JPEG
    is realistically sized; a distinct seed gives a distinct image, which
    defeats prefix caching in cold experiments."""
    if side_px % 28 != 0:
        raise ValueError(f"side_px must be a multiple of 28, got {side_px}")
    rng = np.random.default_rng(seed)
    pixels = rng.integers(0, 256, size=(side_px, side_px, 3), dtype=np.uint8)
    return Image.fromarray(pixels)


def to_data_url(img: Image.Image) -> str:
    """Encode an image as the data-URL string the OpenAI-style API expects."""
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    return f"data:image/jpeg;base64,{b64}"


def text_prompt(target_tokens: int, tokenizer, seed: int) -> str:
    """A unique string that tokenizes to exactly target_tokens tokens
    (measured without special tokens; the chat template adds its own).

    Build over-long text, trim the token ids, decode back — then repair,
    because decode/encode is not a perfect round trip and the count can
    shift by a token."""
    text = f"case {seed}:" + " benchmark" * target_tokens
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) < target_tokens:
        raise ValueError(f"filler too short for {target_tokens} tokens")
    text = tokenizer.decode(ids[:target_tokens])

    for _ in range(50):
        count = len(tokenizer.encode(text, add_special_tokens=False))
        if count == target_tokens:
            return text
        if count > target_tokens:
            ids = tokenizer.encode(text, add_special_tokens=False)[:-1]
            text = tokenizer.decode(ids)
        else:
            text += " ok"
    raise RuntimeError(f"could not converge on {target_tokens} tokens")


def text_payload(model: str, prompt: str, max_tokens: int = 64) -> dict:
    """Request body for a text-only chat completion. temperature=0 so the
    same input always gives the same output (reproducible benchmarks)."""
    return {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0,
        "max_tokens": max_tokens,
    }


def image_payload(model: str, data_url: str, question: str, max_tokens: int = 64,
                  image_first: bool = False) -> dict:
    """Request body for an image + question chat completion.

    image_first matters for prefix caching: the cache reuses a shared
    *prefix*, so repeated content (the image) must come before the part
    that changes (the question) or no request can ever hit the cache."""
    text_block = {"type": "text", "text": question}
    image_block = {"type": "image_url", "image_url": {"url": data_url}}
    content = [image_block, text_block] if image_first else [text_block, image_block]
    return {
        "model": model,
        "messages": [{"role": "user", "content": content}],
        "temperature": 0,
        "max_tokens": max_tokens,
    }

import json
import time

import requests


def send_request(payload: dict, base_url: str = "http://localhost:8000") -> dict:
    """POST one chat completion and stream the reply. Returns timings,
    the server-reported prompt token count, and the full output text."""
    payload = dict(payload, stream=True, stream_options={"include_usage": True})

    t_start = time.perf_counter()
    ttft = None
    parts = []
    usage = None

    url = f"{base_url}/v1/chat/completions"
    with requests.post(url, json=payload, stream=True, timeout=600) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if not line.startswith(b"data: "):
                continue
            data = line[len(b"data: "):]
            if data == b"[DONE]":
                break
            chunk = json.loads(data)
            if chunk.get("usage"):
                usage = chunk["usage"]
            for choice in chunk.get("choices", []):
                content = choice.get("delta", {}).get("content")
                if content:
                    if ttft is None:
                        ttft = time.perf_counter() - t_start
                    parts.append(content)

    return {
        "ttft_s": ttft,
        "e2e_s": time.perf_counter() - t_start,
        "prompt_tokens": usage["prompt_tokens"] if usage else None,
        "completion_tokens": usage["completion_tokens"] if usage else None,
        "output_text": "".join(parts),
    }

import subprocess
import time

import requests

MODEL = "Qwen/Qwen2-VL-2B-Instruct"


QWEN_MM_KWARGS = '{"min_pixels": 784, "max_pixels": 1003520}'
PHI_MM_KWARGS = '{"num_crops": 16}'


def build_args(model: str = MODEL, apc: bool = True, port: int = 8000,
               mm_kwargs: str = QWEN_MM_KWARGS, gpu_mem: float = 0.85,
               trust_remote_code: bool = False) -> list:
    args = [
        "vllm", "serve", model,
        "--dtype", "half",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", str(gpu_mem),
        "--enforce-eager",
        "--limit-mm-per-prompt", '{"image": 1}',
        "--mm-processor-kwargs", mm_kwargs,
        "--port", str(port),
    ]
    if trust_remote_code:
        args.append("--trust-remote-code")
    if not apc:
        args.append("--no-enable-prefix-caching")
    return args


def start(model: str = MODEL, apc: bool = True, port: int = 8000,
          log_path: str = "/content/vllm.log", **kwargs):
    log_file = open(log_path, "w")
    return subprocess.Popen(build_args(model, apc, port, **kwargs),
                            stdout=log_file, stderr=subprocess.STDOUT)


def wait_ready(proc, base_url: str = "http://localhost:8000",
               log_path: str = "/content/vllm.log", timeout_s: int = 600) -> None:
    for _ in range(timeout_s // 5):
        if proc.poll() is not None:
            raise RuntimeError("server exited:\n" + open(log_path).read()[-3000:])
        try:
            if requests.get(f"{base_url}/v1/models", timeout=2).ok:
                return
        except requests.exceptions.ConnectionError:
            pass
        time.sleep(5)
    raise RuntimeError(f"server not ready within {timeout_s}s")


def stop(proc) -> None:
    proc.terminate()
    proc.wait(timeout=30)

import json



def append_jsonl(path: str, record: dict) -> None:
    with open(path, "a") as f:
        f.write(json.dumps(record) + "\n")


def warmup(model: str, tokenizer, base_url: str = "http://localhost:8000") -> None:
    """Throwaway requests so one-time costs (Triton kernel compilation,
    per-shape vision kernels) are paid before any measured request."""
    print("warmup: text")
    send_request(text_payload(model, text_prompt(64, tokenizer, seed=999_000), max_tokens=8), base_url)
    for side in (224, 448, 672, 896):
        print(f"warmup: image {side}px")
        payload = image_payload(model, to_data_url(noise_image(side, seed=999_000 + side)), "Describe.", max_tokens=8)
        send_request(payload, base_url)


def run_text_sweep(model, tokenizer, targets, runs_per_target, out_path,
                   base_url: str = "http://localhost:8000") -> None:
    """EXP-T: unique text prompts at each target token count."""
    seed = 0
    for target in targets:
        for run in range(runs_per_target):
            seed += 1
            prompt = text_prompt(target, tokenizer, seed)
            result = send_request(text_payload(model, prompt, max_tokens=16), base_url)
            append_jsonl(out_path, {
                "experiment": "EXP-T",
                "target_tokens": target,
                "seed": seed,
                "prompt_tokens": result["prompt_tokens"],
                "ttft_s": result["ttft_s"],
                "e2e_s": result["e2e_s"],
            })
            print(f"EXP-T {target:>4} tok  run {run + 1}/{runs_per_target}  "
                  f"prompt_tokens={result['prompt_tokens']}  ttft={result['ttft_s']:.3f}s")


def run_image_sweep(model, sides, runs_per_side, out_path,
                    base_url: str = "http://localhost:8000") -> None:
    """EXP-V: unique noise images at each resolution."""
    seed = 0
    for side in sides:
        for run in range(runs_per_side):
            seed += 1
            data_url = to_data_url(noise_image(side, seed))
            payload = image_payload(model, data_url, "Describe this image.", max_tokens=16)
            result = send_request(payload, base_url)
            append_jsonl(out_path, {
                "experiment": "EXP-V",
                "side_px": side,
                "seed": seed,
                "prompt_tokens": result["prompt_tokens"],
                "ttft_s": result["ttft_s"],
                "e2e_s": result["e2e_s"],
            })
            print(f"EXP-V {side:>3}px  run {run + 1}/{runs_per_side}  "
                  f"prompt_tokens={result['prompt_tokens']}  ttft={result['ttft_s']:.3f}s")


def run_repeated_image(model, side, image_seed, n_questions, out_path, config,
                       image_first, base_url: str = "http://localhost:8000") -> None:
    """EXP-C: one fixed image, many distinct questions. Request 1 is always
    cold; whether 2..n go faster depends on the cache config and on whether
    the image precedes the changing question in the prompt."""
    data_url = to_data_url(noise_image(side, image_seed))
    for i in range(1, n_questions + 1):
        question = f"Question {i}: describe aspect number {i} of this image."
        payload = image_payload(model, data_url, question, max_tokens=16,
                                image_first=image_first)
        result = send_request(payload, base_url)
        append_jsonl(out_path, {
            "experiment": "EXP-C",
            "config": config,
            "request_index": i,
            "side_px": side,
            "prompt_tokens": result["prompt_tokens"],
            "ttft_s": result["ttft_s"],
            "e2e_s": result["e2e_s"],
        })
        print(f"EXP-C [{config}] req {i:>2}/{n_questions}  "
              f"prompt_tokens={result['prompt_tokens']}  ttft={result['ttft_s']:.3f}s")

tokenizer = AutoTokenizer.from_pretrained(PHI, trust_remote_code=True)
print('benchmark code loaded')


In [ ]:
# Launch Phi-3.5-vision (8.4 GB fp16 weights - first download takes a while)
proc = start(model=PHI, mm_kwargs=PHI_MM_KWARGS, gpu_mem=0.9, trust_remote_code=True)
print('server starting, pid', proc.pid)
wait_ready(proc)
print('ready')


## Gate first (fp16 correctness for THIS model), then warmup


In [ ]:
# Gate: coherent caption + record what Phi bills for a 448px image
result = send_request(image_payload(PHI, to_data_url(noise_image(448, seed=0)), 'Describe this image.'))
stripped = result['output_text'].strip()
coherent = len(stripped) >= 10 and '!!!!' not in stripped and len(set(stripped)) > 5
print('caption:', repr(result['output_text'][:200]))
print('prompt_tokens for 448px + question:', result['prompt_tokens'], '(Qwen billed 281)')
print('GATE PASSED' if coherent else 'GATE FAILED - stop and tell Claude')


In [ ]:
warmup(PHI, tokenizer)
print('warmup done')


In [ ]:
# Text sweep (same targets as Qwen run)
run_text_sweep(PHI, tokenizer, targets=[64, 256, 576, 1024], runs_per_target=10, out_path=RESULTS)


In [ ]:
# Image sweep (same pixel sizes; token counts will differ - Phi crops, Qwen tiles)
run_image_sweep(PHI, sides=[224, 448, 672, 896], runs_per_side=10, out_path=RESULTS)


## Analysis: Phi fits, with the Qwen results alongside


In [ ]:
import numpy as np

import json

import numpy as np


def load_jsonl(path: str) -> list[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def fit_line(records: list[dict]) -> tuple[float, float]:
    """Least-squares TTFT vs prompt_tokens.
    Returns (slope in ms per token, intercept in ms)."""
    x = np.array([r["prompt_tokens"] for r in records], dtype=float)
    y = np.array([r["ttft_s"] for r in records], dtype=float) * 1000.0
    slope, intercept = np.polyfit(x, y, 1)
    return float(slope), float(intercept)


def median_ttft_by(records: list[dict], key: str) -> dict:
    """Median TTFT (seconds) per group, e.g. per target_tokens or side_px."""
    groups: dict = {}
    for r in records:
        groups.setdefault(r[key], []).append(r["ttft_s"])
    return {k: float(np.median(v)) for k, v in sorted(groups.items())}


def first_occurrence(records: list[dict], keys: tuple[str, ...]) -> list[dict]:
    """Keep only the first record per key combination. Guards analysis
    against a sweep cell accidentally run twice: repeated inputs are no
    longer cold, so only the first pass counts as a cold measurement."""
    seen = set()
    kept = []
    for r in records:
        key = tuple(r.get(k) for k in keys)
        if key not in seen:
            seen.add(key)
            kept.append(r)
    return kept

QWEN = {'text_slope': 0.2254, 'image_slope': 0.8551, 'ratio': 3.8}  # from results/exp_tv-2026-07-18-meta.json

records = load_jsonl(RESULTS)
text_recs = [r for r in records if r['experiment'] == 'EXP-T']
img_recs = first_occurrence([r for r in records if r['experiment'] == 'EXP-V'], ('side_px', 'seed'))

t_slope, t_int = fit_line(text_recs)
i_slope, i_int = fit_line(img_recs)
print(f'PHI  text : ttft_ms = {t_int:7.1f} + {t_slope:.4f} * prompt_tokens')
print(f'PHI  image: ttft_ms = {i_int:7.1f} + {i_slope:.4f} * prompt_tokens')
print()
print(f'{"":14} {"text ms/100tok":>14} {"image ms/100tok":>15} {"ratio":>6}')
print(f'{"Phi-3.5":14} {t_slope*100:>14.1f} {i_slope*100:>15.1f} {i_slope/t_slope:>6.1f}x')
print(f'{"Qwen2-VL-2B":14} {QWEN["text_slope"]*100:>14.1f} {QWEN["image_slope"]*100:>15.1f} {QWEN["ratio"]:>6.1f}x')
print()
print('phi image tokens by size:', sorted({r['side_px']: r['prompt_tokens'] for r in img_recs}.items()))
print('image medians:', {k: round(v,3) for k,v in median_ttft_by(img_recs, 'side_px').items()})
print('text medians :', {k: round(v,3) for k,v in median_ttft_by(text_recs, 'target_tokens').items()})


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
for recs, color, label in ((text_recs, '#3056C9', 'Phi text'), (img_recs, '#C2410C', 'Phi image (cold)')):
    x = [r['prompt_tokens'] for r in recs]
    y = [r['ttft_s'] * 1000 for r in recs]
    ax.scatter(x, y, s=18, color=color, alpha=0.55, label=label)
    slope, intercept = fit_line(recs)
    xs = np.array([min(x), max(x)], dtype=float)
    ax.plot(xs, intercept + slope * xs, color=color, linewidth=2)
ax.set_xlabel('prompt tokens (server-reported)')
ax.set_ylabel('TTFT (ms)')
ax.set_title('Phi-3.5-vision - TTFT vs prompt tokens (T4 fp16)')
ax.legend(frameon=False)
ax.grid(alpha=0.25)
fig.savefig('/content/phi_ttft.png', dpi=150, bbox_inches='tight')
plt.show()


## Done

Paste the gate output + analysis output back to the Claude session, and download `results_exp_m.jsonl` + `phi_ttft.png` into Downloads.
